# PD02 · Consultar y resumir EVA

[Índice y distribución de las cuatro horas](../ejercicios/README.md) · [Guía de pandas](../docs/manual-pandas.md)

**Tiempo de trabajo: 70 minutos.** El objetivo es traducir una pregunta a filtros verificables y resúmenes. La pregunta de trabajo es: ¿cómo se distribuyen los registros agrícolas de Boyacá entre cultivos durante 2023 y 2024?

| Símbolo | Uso |
|:---:|---|
| 📘 | Concepto o función de apoyo. |
| ✅ | Evidencia que debe quedar visible. |
| ⚠️ | Límite de interpretación. |

Esta plantilla se encuentra en `ejercicios/PD02_consultas_eva.ipynb`. Antes de comenzar, se crea una copia en `soluciones/PD02_consultas_eva.ipynb` dentro del repositorio de cada estudiante. El desarrollo y la entrega se realizan en esa copia; la plantilla de `ejercicios/` se conserva como enunciado. En JupyterLab se puede duplicar el archivo y mover la copia a `soluciones/`, manteniendo el nombre indicado; si ya existe una solución, se continúa sobre ella. Las instrucciones se presentan en celdas Markdown; las celdas de código marcadas como **Desarrollo** contienen el espacio de trabajo. Cada actividad incluye una celda para explicar el resultado.

Se utiliza el kernel del entorno `.venv` del curso. Los CSV ya deben estar descargados mediante `kit/00_datos.py`. La celda inicial prepara las rutas y carga únicamente las entradas de este ejercicio; el desarrollo del análisis corresponde al estudiante. Este notebook se ejecuta de forma independiente.

**Apoyo en el manual:** secciones 4, 5, 8, 10 y las recetas de porcentajes y equivalencia de la sección 11.

## Ubicación de trabajo: la raíz del repositorio

📘 **La raíz del repositorio** es la carpeta principal de la copia del proyecto. En la instalación del curso se llama `big-data-postgraduate` y contiene `README.md`, `kit/`, `ejercicios/` y `soluciones/`:

```text
bigdata/
└── big-data-postgraduate/       ← raíz del repositorio
    ├── README.md
    ├── kit/
    ├── ejercicios/              ← enunciados y plantillas
    └── soluciones/              ← notebooks resueltos
```

**Desde la terminal de Ubuntu en WSL.** `pwd` muestra la carpeta actual, `cd` cambia de carpeta y `ls` muestra su contenido. Los siguientes comandos se ejecutan en la terminal, no en una celda Python:

```bash
pwd
cd /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate
pwd
ls
```

✅ Después de `cd`, el segundo `pwd` debe mostrar `/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate`. En la salida de `ls` deben aparecer `README.md`, `kit`, `ejercicios` y `soluciones`. La carpeta `bigdata` es la carpeta contenedora; la raíz del proyecto es su subcarpeta `big-data-postgraduate`.

La ruta anterior corresponde a la configuración de la guía WSL. Si el repositorio se clonó en otra ubicación, se sustituye por la ruta real de esa copia. Si `cd` informa que la carpeta no existe, se revisa esa ubicación antes de continuar.

**Desde una subcarpeta conocida.** `..` significa «carpeta superior». Se comprueba primero la ubicación con `pwd` y se utiliza la alternativa correspondiente:

| Ubicación actual, relativa al repositorio | Comando para volver a la raíz |
|---|---|
| `ejercicios/`, `soluciones/` o `kit/` | `cd ..` |
| `kit/Notebooks/` | `cd ../..` |

Al llegar a la raíz, se puede activar el entorno y abrir JupyterLab desde la terminal:

```bash
source .venv/bin/activate
jupyter lab
```

**Dentro de un notebook ya abierto.** La terminal y el kernel de Python mantienen sus propios directorios de trabajo. La celda de carga siguiente localiza la raíz y la guarda en `REPO`, aunque el notebook esté abierto desde `soluciones/` o `ejercicios/`. Después de ejecutar esa celda, el siguiente bloque puede ejecutarse en una celda Python para comprobar la ubicación y cambiar el directorio del kernel a la raíz:

```python
import os
from pathlib import Path

print("Directorio antes del cambio:", Path.cwd())
print("Raíz detectada:", REPO)
os.chdir(REPO)
print("Directorio después del cambio:", Path.cwd())
```

✅ `Path.cwd()` debe coincidir con `REPO` después del cambio. Esta operación cambia la base para interpretar rutas relativas; el archivo del notebook permanece en `soluciones/`. La carga de datos de estos ejercicios también funciona sin ese cambio porque construye las rutas a partir de `REPO`.

## Preparación

Se ejecuta la siguiente celda y se comprueba que el intérprete pertenece al entorno del curso y que pandas corresponde a la versión 2.2.3. Las rutas se calculan desde el repositorio, sin introducir rutas personales de Windows, WSL o macOS.

In [7]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

actual = Path.cwd().resolve()
REPO = next(
    (ruta for ruta in [actual, *actual.parents]
     if (ruta / "kit/fuentes.json").is_file()
     and (ruta / "kit/00_datos.py").is_file()),
    None,
)
if REPO is None:
    raise FileNotFoundError("El notebook debe abrirse dentro del repositorio del curso")

RAW = REPO / "kit/data/raw"
OUT = REPO / "kit/salidas/soluciones/PD02"
archivos = ['eva.csv']
faltantes = [nombre for nombre in archivos if not (RAW / nombre).is_file()]
if faltantes:
    raise FileNotFoundError(f"Faltan CSV en {RAW}: {faltantes}")
OUT.mkdir(parents=True, exist_ok=True)

opciones = dict(sep=",", encoding="utf-8-sig", dtype=str, keep_default_na=False)
eva = pd.read_csv(RAW / "eva.csv", **opciones)


print("Python:", sys.version.split()[0])
print("Intérprete:", sys.executable)
print("pandas:", pd.__version__)
print("Repositorio:", REPO)
print("Salidas:", OUT)

Python: 3.12.15
Intérprete: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/.venv/bin/python
pandas: 2.2.3
Repositorio: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate
Salidas: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD02


## 1. Preparación de columnas · 10 minutos

Inspeccionar la tabla y crear `codigo_departamento` a partir de `Código Dane departamento`, `anio` desde `Año`, `area_ha` desde `Área cosechada` y `produccion_t` desde `Producción`. El código se conserva como texto; año, área y producción se convierten a tipos numéricos. Registrar las ausencias resultantes.

📘 **Apoyo:** `.str.strip`, `pd.to_numeric`, `astype("Int64")` e `isna`.

✅ **Comprobación:** Las columnas originales se conservan y no se eliminan filas. El corte de entrada contiene 166.732 registros.

In [9]:
import pandas as pd

# 1. Inspección inicial y control de dimensiones
filas_originales = len(eva)
columnas_originales = list(eva.columns)

print(f"Dimensiones de entrada: {filas_originales:,} filas y {eva.shape[1]} columnas")
assert filas_originales == 166732, f"Se esperaban 166.732 filas, pero se encontraron {filas_originales:,}"

# 2. Creación de 'codigo_departamento' (conserva ceros a la izquierda)
eva['codigo_departamento'] = eva['Código Dane departamento'].astype(str).str.strip()

# 3. Creación de 'anio' (tipo entero nullable Int64)
anio_limpio = eva['Año'].astype(str).str.strip().str.replace(',', '', regex=False)
eva['anio'] = pd.to_numeric(anio_limpio, errors='coerce').astype('Int64')

# 4. Creación de 'area_ha' (flotante con reemplazo de separador decimal)
area_limpia = eva['Área cosechada'].astype(str).str.strip().str.replace(',', '.', regex=False)
eva['area_ha'] = pd.to_numeric(area_limpia, errors='coerce')

# 5. Creación de 'produccion_t' (flotante con reemplazo de separador decimal)
prod_limpia = eva['Producción'].astype(str).str.strip().str.replace(',', '.', regex=False)
eva['produccion_t'] = pd.to_numeric(prod_limpia, errors='coerce')

# 6. Generación del reporte de ausencias resultantes
nuevas = ['codigo_departamento', 'anio', 'area_ha', 'produccion_t']
ausencias = pd.DataFrame({
    'columna_nueva': nuevas,
    'conteo_ausentes': [eva[c].isna().sum() for c in nuevas],
    'pct_ausentes': [f"{(eva[c].isna().sum() / filas_originales) * 100:.2f}%" for c in nuevas]
})

print("\n--- REGISTRO DE AUSENCIAS EN NUEVAS COLUMNAS ---")
display(ausencias)

# 7. Comprobaciones obligatorias (Asserts)
assert len(eva) == 166732, "Error: Se modificó la cantidad de filas del DataFrame original."
assert all(c in eva.columns for c in columnas_originales), "Error: Se eliminaron columnas originales."

print("\n✅ COMPROBACIÓN EXITOSA: Se conservan exactamente 166.732 filas y todas las columnas originales.")

Dimensiones de entrada: 166,732 filas y 22 columnas

--- REGISTRO DE AUSENCIAS EN NUEVAS COLUMNAS ---


,columna_nueva,conteo_ausentes,pct_ausentes
0,codigo_departamento,0,0.00%
1,anio,0,0.00%
2,area_ha,0,0.00%
3,produccion_t,0,0.00%



✅ COMPROBACIÓN EXITOSA: Se conservan exactamente 166.732 filas y todas las columnas originales.


**Interpretación del resultado:**

_Pendiente de completar con evidencia de las tablas o controles anteriores._

## 2. Selección con máscara y loc · 15 minutos

Seleccionar el departamento `15` (Boyacá), los años 2023 y 2024, área cosechada positiva y producción no negativa. Guardar todas las columnas de las filas seleccionadas en `consulta_loc`, utilizando `.copy()`. Presentar el total original, el seleccionado y la diferencia.

📘 **Apoyo:** `.eq`, `.isin`, `.gt`, `.ge`, `&` y `.loc`.

✅ **Comprobación:** Todas las filas seleccionadas cumplen simultáneamente las cuatro condiciones. La diferencia respecto a la entrada incluye registros fuera del alcance territorial o temporal; no implica que todos sean errores.

In [10]:
import pandas as pd

# 1. Definición de las 4 condiciones con los métodos sugeridos
cond_depto = eva['codigo_departamento'].eq('15')        # Departamento 15 (Boyacá)
cond_anio = eva['anio'].isin([2023, 2024])              # Años 2023 y 2024
cond_area = eva['area_ha'].gt(0)                         # Área cosechada estrictamente positiva (> 0)
cond_prod = eva['produccion_t'].ge(0)                   # Producción no negativa (>= 0)

# 2. Combinación de condiciones con el operador binario '&'
mascara_loc = cond_depto & cond_anio & cond_area & cond_prod

# 3. Selección e independencia de memoria mediante .copy()
consulta_loc = eva.loc[mascara_loc].copy()

# 4. Cálculo de totales y diferencia
total_original = len(eva)
total_seleccionado = len(consulta_loc)
diferencia = total_original - total_seleccionado

print("--- RESULTADOS DE LA SELECCIÓN ---")
print(f"Total registros originales:    {total_original:,}")
print(f"Total registros seleccionados: {total_seleccionado:,}")
print(f"Diferencia (filas excluidas):  {diferencia:,}")

# 5. Comprobación de integridad
assert (consulta_loc['codigo_departamento'] == '15').all(), "Error: Existen departamentos distintos a 15."
assert consulta_loc['anio'].isin([2023, 2024]).all(), "Error: Existen años fuera del rango 2023-2024."
assert (consulta_loc['area_ha'] > 0).all(), "Error: Existen registros con área cosechada <= 0 o nula."
assert (consulta_loc['produccion_t'] >= 0).all(), "Error: Existen registros con producción < 0 o nula."

print("\n✅ COMPROBACIÓN EXITOSA: Todas las filas de 'consulta_loc' cumplen las 4 condiciones simultáneamente.")

--- RESULTADOS DE LA SELECCIÓN ---
Total registros originales:    166,732
Total registros seleccionados: 4,924
Diferencia (filas excluidas):  161,808

✅ COMPROBACIÓN EXITOSA: Todas las filas de 'consulta_loc' cumplen las 4 condiciones simultáneamente.


**Interpretación del resultado:**

_Pendiente de completar con evidencia de las tablas o controles anteriores._

## 3. Consulta equivalente con query · 10 minutos

Repetir exactamente la selección anterior mediante `.query`, con `engine="python"`, y guardar el resultado en `consulta_query`. Usar una variable para el código departamental y referenciarla con `@`. Comparar ambos DataFrames sin eliminar columnas.

📘 **Apoyo:** `.query`, `@variable` y `pd.testing.assert_frame_equal`.

✅ **Comprobación:** La comparación debe confirmar las mismas filas, columnas y valores. Si falla, se revisan las condiciones y los tipos antes de continuar.

In [11]:
import pandas as pd

# 1. Definir la variable externa para referenciarla con @
depto_boyaca = "15"

# 2. Construir la consulta equivalente usando .query() con engine="python"
# Mantiene exactamente las mismas condiciones: departamento 15, años 2023-2024, área > 0 y producción >= 0
consulta_query = eva.query(
    'codigo_departamento == @depto_boyaca and anio in [2023, 2024] and area_ha > 0 and produccion_t >= 0',
    engine="python"
).copy()

# 3. Comprobar la equivalencia exacta entre consulta_loc y consulta_query
# pd.testing.assert_frame_equal verifica que ambos DataFrames coincidan en filas, columnas, tipos e índices
pd.testing.assert_frame_equal(consulta_loc, consulta_query)

print("--- RESULTADOS DE LA COMPARACIÓN ---")
print(f"Filas en consulta_loc:   {len(consulta_loc):,}")
print(f"Filas en consulta_query: {len(consulta_query):,}")
print("\n✅ COMPROBACIÓN EXITOSA: Both DataFrames son exactamente idénticos en estructura, tipos y contenido.")

--- RESULTADOS DE LA COMPARACIÓN ---
Filas en consulta_loc:   4,924
Filas en consulta_query: 4,924

✅ COMPROBACIÓN EXITOSA: Both DataFrames son exactamente idénticos en estructura, tipos y contenido.


**Interpretación del resultado:**

_Pendiente de completar con evidencia de las tablas o controles anteriores._

## 4. Conteos y porcentajes · 20 minutos

Sobre `consulta_loc`, construir `resumen` con una fila por año y cultivo y una columna `registros`. Añadir el total de registros de cada año y `porcentaje_anual`. Ordenar los resultados por año y cantidad de registros descendente. Conservar los grupos con claves ausentes mediante `dropna=False`.

📘 **Apoyo:** `groupby`, `size`, `reset_index`, `transform("sum")` y `sort_values`.

✅ **Comprobación:** La suma de registros coincide con el tamaño de consulta_loc. Los porcentajes suman aproximadamente 100 % por año; se admite una pequeña tolerancia numérica y se redondea solo para mostrar.

In [12]:
import pandas as pd

# 1. Agrupar por 'anio' y 'Cultivo' contando filas por grupo y conservando claves ausentes
resumen = (
    consulta_loc.groupby(['anio', 'Cultivo'], dropna=False)
    .size()
    .reset_index(name='registros')
)

# 2. Calcular el total de registros del año correspondiente utilizando transform('sum')
resumen['total_anio'] = resumen.groupby('anio')['registros'].transform('sum')

# 3. Calcular el porcentaje anual de registros por cultivo
resumen['porcentaje_anual'] = (resumen['registros'] / resumen['total_anio']) * 100

# 4. Ordenar resultados por año (ascendente) y registros (descendente)
resumen = resumen.sort_values(by=['anio', 'registros'], ascending=[True, False]).reset_index(drop=True)

# 5. Visualizar los primeros 10 registros formateados
resumen_display = resumen.copy()
resumen_display['porcentaje_anual'] = resumen_display['porcentaje_anual'].map('{:.2f}%'.format)
print("--- RESUMEN DE REGISTROS POR AÑO Y CULTIVO EN BOYACÁ ---")
display(resumen_display.head(10))

# 6. Comprobaciones de conciliación y porcentajes
total_registros_resumen = resumen['registros'].sum()
total_consulta_loc = len(consulta_loc)

print(f"\nTotal registros en 'resumen':     {total_registros_resumen:,}")
print(f"Total registros en 'consulta_loc': {total_consulta_loc:,}")

# Conciliación exacta de filas
assert total_registros_resumen == total_consulta_loc, "Error: La suma de registros no coincide con consulta_loc."

# Comprobación de que la suma de porcentajes da 100% por año (con tolerancia num)
sumas_pct = resumen.groupby('anio')['porcentaje_anual'].sum()
for anio, suma_pct in sumas_pct.items():
    print(f"Suma de porcentajes para {anio}: {suma_pct:.4f}%")
    assert abs(suma_pct - 100.0) < 1e-5, f"Error: Los porcentajes del año {anio} no suman 100%."

print("\n✅ COMPROBACIÓN EXITOSA: Los conteos concilian perfectamente y los porcentajes suman 100% por año.")

--- RESUMEN DE REGISTROS POR AÑO Y CULTIVO EN BOYACÁ ---


,anio,Cultivo,registros,total_anio,porcentaje_anual
0,2023,Papa,239,2436,9.81%
1,2023,Maíz,194,2436,7.96%
2,2023,Arveja,176,2436,7.22%
3,2023,Frijol,162,2436,6.65%
4,2023,Tomate,113,2436,4.64%
5,2023,Cebolla de bulbo,93,2436,3.82%
6,2023,Yuca,82,2436,3.37%
7,2023,Caña,63,2436,2.59%
8,2023,Trigo,58,2436,2.38%
9,2023,Mora,51,2436,2.09%



Total registros en 'resumen':     4,924
Total registros en 'consulta_loc': 4,924
Suma de porcentajes para 2023: 100.0000%
Suma de porcentajes para 2024: 100.0000%

✅ COMPROBACIÓN EXITOSA: Los conteos concilian perfectamente y los porcentajes suman 100% por año.


**Interpretación del resultado:**

_Pendiente de completar con evidencia de las tablas o controles anteriores._

## 5. Tabla dinámica y conclusiones · 15 minutos

Construir una tabla dinámica a partir de `resumen`, con cultivos en filas, años en columnas y conteos como valores. Completar con cero las combinaciones sin registros. Exportar `resumen` a `OUT / "resumen_eva.csv"`, sin índice, y redactar dos conclusiones.

📘 **Apoyo:** `pivot_table`, `fill_value=0` y `to_csv`.

✅ **Comprobación:** Los totales de la tabla dinámica concilian con resumen. Las conclusiones se refieren a la cantidad de registros y no a la participación de cada cultivo en la superficie agrícola.

In [13]:
import pandas as pd

# 1. Construir la tabla dinámica a partir del DataFrame 'resumen'
# Cultivos en filas, años en columnas y conteo de registros como valores
tabla_dinamica = pd.pivot_table(
    resumen,
    index='Cultivo',
    columns='anio',
    values='registros',
    aggfunc='sum',
    fill_value=0
)

# Visualizar la tabla dinámica
print("--- TABLA DINÁMICA DE REGISTROS POR CULTIVO Y AÑO EN BOYACÁ ---")
display(tabla_dinamica.head(15))

# 2. Conciliación de totales
total_resumen = resumen['registros'].sum()
total_tabla_dinamica = tabla_dinamica.sum().sum()

print(f"\nTotal registros en 'resumen':        {total_resumen:,}")
print(f"Total registros en 'tabla_dinamica': {total_tabla_dinamica:,}")

# Comprobación de equivalencia
assert total_resumen == total_tabla_dinamica, "Error: Los totales no coinciden entre resumen y la tabla dinámica."
print("✅ COMPROBACIÓN EXITOSA: Los totales concilian perfectamente.")

# 3. Exportar resumen a la carpeta OUT sin índice
ruta_salida = OUT / "resumen_eva.csv"
resumen.to_csv(ruta_salida, index=False)
print(f"\n✅ Archivo exportado exitosamente en: {ruta_salida}")

--- TABLA DINÁMICA DE REGISTROS POR CULTIVO Y AÑO EN BOYACÁ ---


anio,2023,2024
Cultivo,,
Acelga,5,8
Achira,4,4
Agraz - mortiño,3,4
Aguacate,48,52
Ahuyama,23,26
Ajo,7,7
Ají,1,2
Apio,0,1
Arazá,3,4



Total registros en 'resumen':        4,924
Total registros en 'tabla_dinamica': 4,924
✅ COMPROBACIÓN EXITOSA: Los totales concilian perfectamente.

✅ Archivo exportado exitosamente en: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD02/resumen_eva.csv


**Interpretación del resultado:**

_Pendiente de completar con evidencia de las tablas o controles anteriores._

## Entrega

El notebook completado se conserva como `soluciones/PD02_consultas_eva.ipynb`. La salida regenerable es `kit/salidas/soluciones/PD02/resumen_eva.csv`.

✅ Deben quedar visibles la equivalencia de consultas, la conciliación de los conteos, los porcentajes por año, la tabla dinámica y dos conclusiones.

⚠️ Una fila EVA es una observación de la fuente, no un predio ni una persona. Un porcentaje de registros no equivale a un porcentaje de área cultivada. El cero de una tabla de conteos indica que no se encontraron filas para esa combinación.